# Disney Pin Catalog Parser — Google Colab GPU Runner

This notebook clones the public GitHub repository, installs the parser, starts Ollama for the default GPU vision extraction path unless disabled, runs the existing `run_parser.sh`, saves the final Excel workbook to Google Drive, and downloads a copy.

**Runtime:** In Colab choose **Runtime → Change runtime type → GPU** before running all cells for the default Ollama vision mode. Set `USE_OLLAMA_VISION = False` for a faster deterministic-only run without Ollama; this sets `PIN_USE_OLLAMA=0`.


In [ ]:
#@title 1. Configure the public repo, run mode, and Google Drive output
REPO_URL = "https://github.com/YOUR-USERNAME/pinproject.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_OLLAMA_VISION = True  #@param {type:"boolean"}
OLLAMA_MODEL = "qwen2.5vl:3b"  #@param {type:"string"}
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/pinproject-output"  #@param {type:"string"}

# Ollama vision is the default and can improve difficult pages.
# Turn USE_OLLAMA_VISION off for a faster deterministic-only run; the notebook will set PIN_USE_OLLAMA=0.


In [ ]:
#@title 2. Check GPU and clone/update the repository
import os, shutil, subprocess
from pathlib import Path

workdir = Path('/content/pinproject')
if workdir.exists():
    shutil.rmtree(workdir)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(workdir)], check=True)
os.chdir(workdir)
print('Repository ready at', workdir)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
#@title 3. Mount Google Drive and choose output folder
from pathlib import Path
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = Path(DRIVE_OUTPUT_DIR)
else:
    output_dir = Path('/content/pinproject/output')
output_dir.mkdir(parents=True, exist_ok=True)
print('Workbook will be saved to:', output_dir / 'disney-pin-catalogue.xlsx')


In [ ]:
#@title 4. Install Python dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', '.'], check=True)
print('Python package installed')


In [ ]:
#@title 5. Install/start Ollama unless deterministic-only mode is selected
import os, subprocess, time, shutil

if USE_OLLAMA_VISION:
    if shutil.which('ollama') is None:
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
    server = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    time.sleep(8)
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    os.environ['PIN_USE_OLLAMA'] = '1'
    os.environ['PIN_OLLAMA_MODEL'] = OLLAMA_MODEL
    os.environ['PIN_OLLAMA_TIMEOUT'] = '1800'
    print('Ollama vision mode ready with', OLLAMA_MODEL)
else:
    os.environ['PIN_USE_OLLAMA'] = '0'
    print('Fast deterministic parser selected; PIN_USE_OLLAMA=0, skipping Ollama install.')


In [ ]:
#@title 6. Add or inspect PDFs
from pathlib import Path
pdf_dir = Path('catalog-pdfs')
pdf_dir.mkdir(exist_ok=True)
pdfs = sorted(pdf_dir.glob('*.pdf')) + sorted(pdf_dir.glob('*.PDF'))
print(f'Found {len(pdfs)} PDFs in {pdf_dir}:')
for pdf in pdfs:
    print(' -', pdf)

# If your public repo does not include PDFs, upload them into /content/pinproject/catalog-pdfs
# with the Colab file browser, then rerun this cell.


In [ ]:
#@title 7. Run the existing shell file and create the Excel output
import os, subprocess
env = os.environ.copy()
env['PYTHON_BIN'] = 'python3'
env['OUTPUT_DIR'] = str(output_dir)
subprocess.run(['bash', 'run_parser.sh'], check=True, env=env)


In [ ]:
#@title 8. Download the final workbook
from google.colab import files
workbook = output_dir / 'disney-pin-catalogue.xlsx'
if not workbook.exists():
    raise FileNotFoundError(workbook)
print('Final workbook:', workbook, workbook.stat().st_size, 'bytes')
files.download(str(workbook))
